In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1]))  # adds teehr-fved/ to path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates

In [ ]:
from riverware.postprocessing.load_results import run_postprocessing

MODEL_DIR = Path(r"C:\FVED\Models\CRMMS-ESP\CRMMS-March2025")
CONFIGURATION_NAME = "crmms_esp_march2025"

df = run_postprocessing(model_dir=MODEL_DIR, configuration_name=CONFIGURATION_NAME)
print(f"Loaded {len(df):,} rows")

In [ ]:
df.tail()

## Schema & Shape

In [ ]:
print("Shape:", df.shape)
print("\nDtypes:")
print(df.dtypes)
df.head()

## Coverage: Variables, Locations, Members, Time Range

In [ ]:
print("reference_time:", df["reference_time"].iloc[0])
print("value_time range:", df["value_time"].min(), "→", df["value_time"].max())
print(f"\nMembers ({df['member'].nunique()}):", sorted(df["member"].unique(), key=int))
print(f"\nLocations ({df['location_id'].nunique()}):")
print(df["location_id"].unique())
print(f"\nVariables ({df['variable_name'].nunique()}):")
print(sorted(df["variable_name"].unique()))

## Missing Values & Duplicates

In [ ]:
print("Null counts:")
print(df.isnull().sum())
print(f"\nNaN values: {df['value'].isna().sum():,} ({df['value'].isna().mean():.1%})")
print(f"Duplicate rows: {df.duplicated().sum():,}")

# NaN breakdown by variable
nan_by_var = (
    df[df["value"].isna()]
    .groupby("variable_name")
    .size()
    .sort_values(ascending=False)
)
if not nan_by_var.empty:
    print("\nNaN counts by variable_name:")
    print(nan_by_var.to_string())

## Value Summary by Variable

In [ ]:
df.groupby(["variable_name", "unit_name"])["value"].describe().round(2)

## Helpers

Reusable functions for filtering the DataFrame and building the standard CRMMS plots used in the Excel workbook.

In [ ]:
def get_slot(df: pd.DataFrame, object_name: str, slot_name: str, prefix: str = "crmms") -> pd.DataFrame:
    """Return rows for a single RiverWare object.slot combination."""
    return df[
        (df["location_id"] == f"{prefix}-{object_name}") &
        (df["variable_name"] == slot_name)
    ]


def spaghetti_fan_plot(slot_df: pd.DataFrame, title: str, ylabel: str,
                        hlines: dict | None = None, ax=None):
    """Plot all traces as thin lines with a 10/25/50/75/90th-percentile fan overlay.

    hlines: {label: value} for horizontal reference lines (e.g. shortage thresholds).
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(13, 4))
    else:
        fig = ax.figure

    for _, grp in slot_df.groupby("member"):
        ax.plot(grp["value_time"], grp["value"], color="steelblue", alpha=0.25, linewidth=0.7)

    pct = slot_df.groupby("value_time")["value"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).unstack()
    ax.fill_between(pct.index, pct[0.1], pct[0.9], alpha=0.15, color="steelblue", label="10–90th pct")
    ax.fill_between(pct.index, pct[0.25], pct[0.75], alpha=0.3, color="steelblue", label="25–75th pct")
    ax.plot(pct.index, pct[0.5], color="navy", linewidth=1.8, label="Median")

    if hlines:
        colors = plt.cm.tab10.colors
        for i, (label, val) in enumerate(hlines.items()):
            ax.axhline(val, linestyle="--", linewidth=1, color=colors[i % 10], label=label)

    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
    ax.legend(fontsize=8, loc="best")
    fig.autofmt_xdate(rotation=0, ha="center")
    plt.tight_layout()
    return fig, ax


def exceedance_curve(slot_df: pd.DataFrame, target_date, title: str, xlabel: str, ax=None):
    """Plot exceedance probability vs. value at a specific value_time (mirrors Excel Exceedance sheet)."""
    subset = slot_df[slot_df["value_time"] == pd.Timestamp(target_date, tz="UTC")]
    vals = subset["value"].dropna().sort_values(ascending=False).reset_index(drop=True)
    # Weibull plotting position
    probs = (np.arange(1, len(vals) + 1)) / (len(vals) + 1)

    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 4))
    else:
        fig = ax.figure

    ax.plot(vals, probs, marker="o", markersize=4, linewidth=1.2)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Probability of Exceedance")
    ax.set_title(f"{title}\n{pd.Timestamp(target_date).strftime('%B %Y')}")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    return fig, ax


## Powell Pool Elevation

Mirrors the `Powell.Pool Elevation` sheet in the Excel workbook. Reference lines match key CRMMS operational tiers.

In [ ]:
powell_elev = get_slot(df, "Powell", "Pool Elevation")

powell_hlines = {
    "Min Power Pool (3,490 ft)": 3490,
    "LBDV Trigger (3,525 ft)":   3525,
    "Equalization (3,575 ft)":   3575,
}

spaghetti_fan_plot(powell_elev, "Lake Powell — Pool Elevation (ft)", "Elevation (ft)", hlines=powell_hlines)
plt.show()

## Mead Pool Elevation

Mirrors the `Mead.Pool Elevation` sheet. Dashed reference lines show DCP shortage tiers.

In [ ]:
mead_elev = get_slot(df, "Mead", "Pool Elevation")

mead_hlines = {
    "Tier 1 Shortage (1,025 ft)": 1025,
    "Tier 2 Shortage (1,000 ft)": 1000,
    "Tier 3 Shortage (  950 ft)":  950,
    "Dead Pool        (  895 ft)":  895,
}

spaghetti_fan_plot(mead_elev, "Lake Mead — Pool Elevation (ft)", "Elevation (ft)", hlines=mead_hlines)
plt.show()

## Exceedance Curves — Powell & Mead

Mirrors the `Exceedance` sheet. Change `TARGET_DATE` to any monthly timestep in the simulation period.

In [ ]:
TARGET_DATE = "2026-01-01"  # change to any YYYY-MM-DD in the simulation period

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
exceedance_curve(powell_elev, TARGET_DATE, "Lake Powell Pool Elevation", "Elevation (ft)", ax=ax1)
exceedance_curve(mead_elev,   TARGET_DATE, "Lake Mead Pool Elevation",   "Elevation (ft)", ax=ax2)
plt.show()
